# Predicción del Tipo de Cambio USD/GBP con MLP

En este notebook aplicamos un **Perceptrón Multicapa (MLP)** para predecir el tipo de cambio USD/GBP, siguiendo la estructura del notebook de referencia sobre redes neuronales recurrentes.

## 1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

## 2. Carga y preprocesamiento de datos

En el notebook de referencia se generan series temporales sintéticas. Aquí, en cambio, cargamos datos reales del tipo de cambio USD/GBP.

In [ ]:
RUTA = 'USD_GBP Historical Data.csv'
df = pd.read_csv(RUTA)
print(f'Dimensiones del dataset: {df.shape}')
df.head(10)

In [ ]:
# Convertir la columna Date a datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Convertir columnas numéricas
for col in ['Price', 'Open', 'High', 'Low']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ordenar por fecha (de más antigua a más reciente)
df = df.sort_values('Date').reset_index(drop=True)

# Usar solo el precio de cierre (Price)
prices = df['Price'].values.astype(np.float32)
print(f'Total de datos: {len(prices)}')
print(f'Rango de fechas: {df["Date"].iloc[0]} a {df["Date"].iloc[-1]}')

## 3. Creación de secuencias (series temporales)

Similar al notebook de referencia, creamos secuencias de `n_steps` valores para predecir el siguiente valor.
La función `generate_time_series` del notebook de referencia genera datos sintéticos; aquí extraemos secuencias reales del CSV.

In [ ]:
n_steps = 50  # Mismo valor que en el notebook de referencia

# Normalizar los datos (Min-Max scaling a [0, 1])
price_min = prices.min()
price_max = prices.max()
prices_norm = (prices - price_min) / (price_max - price_min)

# Crear secuencias: cada muestra tiene n_steps+1 valores (n_steps de entrada + 1 target)
def create_sequences(data, n_steps):
    X = []
    y = []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    # Añadir dimensión de feature (como en el notebook de referencia)
    X = X[..., np.newaxis]  # shape: (num_samples, n_steps, 1)
    y = y[..., np.newaxis]  # shape: (num_samples, 1)
    return X, y

X, y = create_sequences(prices_norm, n_steps)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 4. División en conjuntos de entrenamiento, validación y prueba

Siguiendo la estructura del notebook de referencia, dividimos en train / valid / test (70% / 15% / 15%), manteniendo el orden temporal.

In [ ]:
total = len(X)
train_end = int(total * 0.70)
valid_end = int(total * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
X_test, y_test = X[valid_end:], y[valid_end:]

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Validación:    {X_valid.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')

## 5. Visualización de las series

Función `plot_series` adaptada del notebook de referencia para visualizar las predicciones.

In [ ]:
def plot_series(series, y=None, y_pred=None, x_label="$t$", y_label="$x$", title=None):
    """Función adaptada del notebook de referencia para visualizar series y predicciones."""
    r, c = 3, 5
    fig, axes = plt.subplots(nrows=r, ncols=c, sharey=True, sharex=True, figsize=(20, 10))
    if title:
        fig.suptitle(title, fontsize=16)
    for row in range(r):
        for col in range(c):
            plt.sca(axes[row][col])
            ix = col + row * c
            if ix >= len(series):
                continue
            plt.plot(series[ix, :, 0], ".-")
            if y is not None:
                plt.plot(len(series[ix, :, 0]), y[ix, 0], "bx", markersize=10)
            if y_pred is not None:
                plt.plot(len(series[ix, :, 0]), y_pred[ix, 0], "ro")
            plt.grid(True)
            if x_label and row == r - 1:
                plt.xlabel(x_label, fontsize=12)
            if y_label and col == 0:
                plt.ylabel(y_label, fontsize=12, rotation=0)
    plt.tight_layout()
    plt.show()

# Visualizar algunas series del conjunto de prueba
plot_series(X_test[:15], y_test[:15], title='Series del conjunto de prueba (valores reales)')

## 6. Predicción base (Baseline)

Igual que en el notebook de referencia, el baseline más simple para series temporales es **predecir que el siguiente valor será igual al último valor observado** (predicción naive).

In [ ]:
# Baseline: predecir el ultimo valor de la secuencia
y_pred_baseline = X_test[:, -1, :]  # Ultimo paso temporal de cada secuencia

mse_baseline = np.mean((y_test - y_pred_baseline) ** 2)
print(f'MSE Baseline (naive): {mse_baseline:.6f}')

plot_series(X_test[:15], y_test[:15], y_pred_baseline[:15], 
            title='Baseline: Predicción Naive (último valor = predicción)')

## 7. Modelo MLP (Perceptrón Multicapa) con PyTorch

Siguiendo la estructura del notebook de referencia, implementamos un **MLP** usando PyTorch (`nn.Module`). El MLP toma como entrada una secuencia de `n_steps` valores (aplanados) y predice el siguiente valor.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_steps, n_outputs=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(n_steps, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_outputs)
        )

    def forward(self, x):
        x = self.flatten(x)  # (batch, n_steps*1) -> aplanar la dimensión temporal y features
        return self.fc(x)

model = MLP(n_steps)
print(model)

## 8. Entrenamiento del modelo

Usamos **MSE** como función de pérdida y **Adam** como optimizador, igual que en el notebook de referencia.

In [ ]:
# Convertir a tensores de PyTorch
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_valid_t = torch.from_numpy(X_valid)
y_valid_t = torch.from_numpy(y_valid)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

# Hiperparámetros
lr = 0.001
epochs = 100
batch_size = 64

# Función de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# DataLoaders
train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Entrenamiento
train_losses = []
valid_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    train_loss = epoch_loss / n_batches
    train_losses.append(train_loss)

    # Pérdida de validación
    model.eval()
    with torch.no_grad():
        y_valid_pred = model(X_valid_t)
        valid_loss = criterion(y_valid_pred, y_valid_t).item()
        valid_losses.append(valid_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.6f} | Valid Loss: {valid_loss:.6f}')

print(f'\nEntrenamiento finalizado.')
print(f'Loss final (train): {train_losses[-1]:.6f}')
print(f'Loss final (valid): {valid_losses[-1]:.6f}')

## 9. Curva de pérdida

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(valid_losses, label='Valid Loss')
plt.title('Curva de Pérdida (MSE) durante el Entrenamiento', fontsize=14)
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Evaluación en el conjunto de prueba

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_mlp = model(X_test_t).numpy()

mse_mlp = np.mean((y_test - y_pred_mlp) ** 2)

print('=== Comparación de MSE ===')
print(f'MSE Baseline (naive):  {mse_baseline:.6f}')
print(f'MSE MLP:               {mse_mlp:.6f}')

if mse_mlp < mse_baseline:
    mejora = (1 - mse_mlp / mse_baseline) * 100
    print(f'\n✅ El MLP mejora el baseline en {mejora:.1f}%')
else:
    print(f'\n❌ El MLP no supera el baseline')

## 11. Visualización de predicciones (estilo notebook de referencia)

Usamos la función `plot_series` adaptada del notebook de referencia para comparar las predicciones del MLP contra los valores reales.

In [ ]:
# Visualizar predicciones del MLP vs valores reales en el conjunto de prueba
plot_series(X_test[:15], y_test[:15], y_pred_mlp[:15],
            title='MLP: Predicción vs Valor Real (conjunto de prueba)')

In [ ]:
# Desnormalizar para gráfica en escala real
def denormalize(data, pmin, pmax):
    return data * (pmax - pmin) + pmin

y_test_real = denormalize(y_test[:, 0], price_min, price_max)
y_pred_real = denormalize(y_pred_mlp[:, 0], price_min, price_max)
y_base_real = denormalize(y_pred_baseline[:, 0], price_min, price_max)

# Gráfica de predicciones vs valores reales (serie completa del test)
plt.figure(figsize=(14, 5))
plt.plot(y_test_real, label='Valor Real', color='steelblue', linewidth=1)
plt.plot(y_pred_real, label='Predicción MLP', color='crimson', linewidth=1, linestyle='--')
plt.plot(y_base_real, label='Baseline (naive)', color='gray', linewidth=1, linestyle=':')
plt.title('Predicción vs Valor Real - Conjunto de Prueba (escala original)', fontsize=14)
plt.xlabel('Muestras')
plt.ylabel('Precio USD/GBP')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica de dispersión: predicción vs real
plt.figure(figsize=(6, 6))
plt.scatter(y_test_real, y_pred_real, alpha=0.5, s=10, color='steelblue', label='MLP')
min_val = min(y_test_real.min(), y_pred_real.min())
max_val = max(y_test_real.max(), y_pred_real.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1, label='Ideal')
plt.title('Dispersión: Valor Real vs Predicción', fontsize=14)
plt.xlabel('Valor Real')
plt.ylabel('Predicción')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Conclusiones

- Siguiendo la estructura del notebook de referencia, implementamos un **MLP** con PyTorch para predecir el tipo de cambio USD/GBP.
- Se utilizó una ventana de **50 pasos temporales** (`n_steps=50`) para predecir el siguiente valor.
- Se estableció un **baseline naive** (predecir el último valor observado) como punto de comparación.
- La métrica de evaluación es el **MSE** (Error Cuadrático Medio), igual que en el notebook de referencia.
- Se compararon las predicciones del MLP con el baseline para determinar si la red neuronal aporta una mejora significativa.